In [2]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/raw/listings.csv', encoding='utf-8')
df.shape
df.head(5)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2539,https://www.airbnb.com/rooms/2539,20260614073253,2026-06-15,city scrape,11 Min to Manhattan • Prospect Park • Fast WiFi,"Bright, serene room in a renovated apartment h...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,2787,...,5.00,4.78,4.78,NaN,NaN,5,0,5,0,0.08
1,6848,https://www.airbnb.com/rooms/6848,20260614073253,2026-06-14,city scrape,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,https://a0.muscache.com/pictures/e4f031a7-f146...,15991,...,4.80,4.69,4.59,NaN,NaN,1,1,0,0,0.95
2,6872,https://www.airbnb.com/rooms/6872,20260614073253,2026-06-14,city scrape,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,16104,...,5.00,5.00,5.00,NaN,NaN,2,0,2,0,0.04
3,6990,https://www.airbnb.com/rooms/6990,20260614073253,2026-06-14,city scrape,UES Beautiful Blue Room,Beautiful peaceful healthy home,NaN,https://a0.muscache.com/pictures/45fb4ec7-6856...,16800,...,4.94,4.85,4.84,NaN,NaN,1,0,1,0,1.24
4,7097,https://www.airbnb.com/rooms/7097,20260614073253,2026-06-14,city scrape,"Perfect for Your Parents, With Garden & Patio",Parents/grandparents coming to town or are you...,NaN,https://a0.muscache.com/pictures/aaac19fc-4b4d...,17571,...,4.93,4.95,4.82,NaN,NaN,2,0,2,0,2.12


In [5]:
df.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_profile_id', 'host_profile_url', 'host_name',
       'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months',
       'hosts_time_as_host_years', 'hosts_time_as_host_months',
       'host_location', 'host_about', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
       'host_listings_count', 'host_total_listings_count',
       'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'price_quote_checkin_date', 'price_quote_c

In [97]:
df.shape

(30259, 90)

## Requirements for data cleaning
`host_identity_verified` should be t,
`price_quote_price_per_night` should not be NA,
`has_availability` should be t,
active referring to `estimated_occupancy_l365d` not 0 or `availability_365` (availability for next 365 days) not 0




In [259]:
verified = df['host_identity_verified'] == 't'
has_price = df['price_quote_price_per_night'].notna()
available = df['has_availability'] == 't'
active = ((df['estimated_occupancy_l365d'] != 0) | (df['availability_365'] != 0))

df_clean = df[verified & has_price & available & active]
df_clean.shape

(20810, 90)

In [265]:
df_clean.loc[df_clean['maximum_nights'] < df_clean['minimum_nights'], ['accommodates', 'beds', 'bedrooms']]

,accommodates,beds,bedrooms


In [282]:
df_clean['review_scores_rating'].describe()

count    15061.000000
mean         4.726586
std          0.459026
min          1.000000
25%          4.670000
50%          4.860000
75%          5.000000
max          5.000000
Name: review_scores_rating, dtype: float64

In [286]:
df_analysis = df_clean[['id', 'name', 'host_id', 'host_name', 'host_is_superhost', 'host_listings_count', 'neighbourhood_cleansed', 
    'neighbourhood_group_cleansed', 'latitude', 'longitude', 'room_type', 'accommodates', 'price_quote_checkin_date', 'price_quote_checkout_date',
    'price_quote_price_per_night', 'minimum_nights', 'number_of_reviews', 'number_of_reviews_l30d', 'review_scores_rating', 
    'reviews_per_month', 'availability_30', 'calendar_last_scraped', 'calculated_host_listings_count', 'estimated_occupancy_l365d', 
    'estimated_revenue_l365d']]

df_analysis


,id,name,host_id,host_name,host_is_superhost,host_listings_count,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,...,minimum_nights,number_of_reviews,number_of_reviews_l30d,review_scores_rating,reviews_per_month,availability_30,calendar_last_scraped,calculated_host_listings_count,estimated_occupancy_l365d,estimated_revenue_l365d
0,2539,11 Min to Manhattan • Prospect Park • Fast WiFi,2787,John,f,5.0,Kensington,Brooklyn,40.645937,-73.972164,...,30.0,10,1,4.80,0.08,4,2026-06-15,5,60,6838.0
1,6848,Only 2 stops to Manhattan studio,15991,Allen,t,1.0,Williamsburg,Brooklyn,40.709350,-73.953420,...,30.0,198,0,4.60,0.95,1,2026-06-14,1,180,21109.0
2,6872,Uptown Sanctuary w/ Private Bath (Month to Month),16104,Kae,f,2.0,East Harlem,Manhattan,40.801070,-73.942550,...,30.0,2,0,5.00,0.04,0,2026-06-14,2,60,4804.0
3,6990,UES Beautiful Blue Room,16800,Cynthia,t,1.0,East Harlem,Manhattan,40.787780,-73.947590,...,30.0,251,0,4.88,1.24,0,2026-06-14,1,120,9260.0
4,7097,"Perfect for Your Parents, With Garden & Patio",17571,Jane,t,2.0,Fort Greene,Brooklyn,40.691940,-73.973890,...,30.0,423,0,4.89,2.12,0,2026-06-14,2,255,51630.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30254,1706698360105939193,Renovated UES Yorkville Studio | W/D in unit,27181542,JT Urban,f,69.0,Upper East Side,Manhattan,40.776070,-73.948900,...,30.0,0,0,NaN,NaN,27,2026-06-14,71,0,0.0
30255,1706706813962519891,Cozy and fun light-filled house in Park Slope,12153770,Alice,f,1.0,South Slope,Brooklyn,40.664271,-73.984940,...,30.0,0,0,NaN,NaN,0,2026-06-14,1,0,0.0
30256,1706709939753623262,2BD Apartment in South Harlem,247439246,Charlene,f,6.0,Harlem,Manhattan,40.801140,-73.954090,...,30.0,0,0,NaN,NaN,13,2026-06-14,7,0,0.0
30257,1706773135420910096,The Grand Waterfront Suite,686962449,Boshuo,t,11.0,Long Island City,Queens,40.739423,-73.961239,...,30.0,0,0,NaN,NaN,27,2026-06-14,2,0,0.0


In [287]:
df_analysis['price_quote_checkin_date'] = pd.to_datetime(df_analysis['price_quote_checkin_date'], format='%Y-%m-%d')
df_analysis['price_quote_checkin_date'].head(5)

0   2026-07-11
1   2026-12-20
2   2026-09-01
3   2026-12-06
4   2027-02-28
Name: price_quote_checkin_date, dtype: datetime64[us]

In [302]:
df_analysis.groupby('neighbourhood_group_cleansed').describe()

id                                       \
                               count          mean       min          25%   
neighbourhood_group_cleansed                                                
Bronx                          820.0  6.517768e+17   45935.0  45267870.75   
Brooklyn                      7133.0  6.604076e+17    2539.0  36633354.00   
Manhattan                     8993.0  7.943206e+17    6872.0  46430844.00   
Queens                        3569.0  7.101290e+17   12937.0  44699042.00   
Staten Island                  295.0  6.117999e+17  225976.0  43055790.50   

                                                                        \
                                       50%           75%           max   
neighbourhood_group_cleansed                                             
Bronx                         7.242343e+17  1.028431e+18  1.704868e+18   
Brooklyn                      7.181136e+17  1.185157e+18  1.706707e+18   
Manhattan                     9.203224e+17  1.329976e+18  1.706710e+18   
Queens                        7.625800e+17  1.217657e+18  1.707075e+18   
Staten Island                 6.718524e+17  1.121797e+18  1.689495e+18   

                                           host_id                ...  \
                                       std   count          mean  ...   
neighbourhood_group_cleansed                                      ...   
Bronx                         5.590761e+17   820.0  2.642558e+08  ...   
Brooklyn                      6.113634e+17  7133.0  3.298603e+15  ...   
Manhattan                     6.266300e+17  8993.0  2.998398e+15  ...   
Queens                        5.953553e+17  3569.0  7.098081e+15  ...   
Staten Island                 5.860542e+17   295.0  1.711907e+16  ...   

                             estimated_revenue_l365d                \
                                                 max           std   
neighbourhood_group_cleansed                                         
Bronx                                       171274.0  17045.748156   
Brooklyn                                   1178610.0  36484.077685   
Manhattan                                  1788741.0  56187.366931   
Queens                                      584205.0  25964.138270   
Staten Island                               165704.0  18556.935424   

                             checkin_week                              \
                                    count                        mean   
neighbourhood_group_cleansed                                            
Bronx                                 820  2026-07-20 03:41:16.097561   
Brooklyn                             7133  2026-07-28 19:55:31.501472   
Manhattan                            8993  2026-07-27 17:17:55.636606   
Queens                               3569  2026-07-17 02:08:42.499299   
Staten Island                         295  2026-07-18 09:50:38.644067   

                                                                        \
                                              min                  25%   
neighbourhood_group_cleansed                                             
Bronx                         2026-06-08 00:00:00  2026-06-15 00:00:00   
Brooklyn                      2026-06-08 00:00:00  2026-06-15 00:00:00   
Manhattan                     2026-06-08 00:00:00  2026-06-15 00:00:00   
Queens                        2026-06-08 00:00:00  2026-06-15 00:00:00   
Staten Island                 2026-06-15 00:00:00  2026-06-15 00:00:00   

                                                                        \
                                              50%                  75%   
neighbourhood_group_cleansed                                             
Bronx                         2026-06-22 00:00:00  2026-08-03 00:00:00   
Brooklyn                      2026-06-29 00:00:00  2026-08-10 00:00:00   
Manhattan                     2026-07-06 00:00:00  2026-08-10 00:00:00   
Queens                        2026-06-22 00:00:00  2026-

In [298]:
# Creating a week aggregation
df_analysis['checkin_week'] = df_analysis['price_quote_checkin_date'].dt.to_period('W').dt.start_time
df_analysis['checkin_week'].value_counts().sort_values(ascending=False)

checkin_week
2026-06-15    5656
2026-06-08    2269
2026-06-29    1808
2026-07-27    1631
2026-06-22    1177
2026-08-03     853
2026-08-31     803
2026-07-20     719
2026-07-06     718
2026-08-10     703
2026-07-13     590
2026-12-28     466
2026-09-28     449
2026-08-17     378
2026-10-26     296
2026-08-24     257
2026-09-14     163
2026-09-07     158
2026-11-30     153
2026-12-14     140
2026-09-21     135
2026-12-21     117
2026-11-02     113
2026-10-05      98
2026-10-19      78
2026-10-12      75
2026-11-09      72
2027-01-04      70
2026-11-16      67
2027-02-01      59
2027-03-29      56
2026-11-23      54
2027-04-26      49
2026-12-07      49
2027-01-25      49
2027-03-01      46
2027-01-11      35
2027-05-10      29
2027-02-22      26
2027-03-22      20
2027-01-18      18
2027-02-08      18
2027-03-08      16
2027-04-19      12
2027-04-05      11
2027-02-15      10
2027-04-12       9
2027-03-15       8
2027-05-17       7
2027-05-03       7
2027-05-24       6
2027-05-31       2

In [297]:
# creating a function for week aggregation:
def get_week(df, date_col = 'price_quote_checkin_date'):
    df['checkin_week'] = pd.to_datetime(df[date_col]).dt.to_period('W').dt.start_time

In [296]:
df_analysis.groupby('checkin_week').agg(
    ave_price=('price_quote_price_per_night', 'mean')
).sort_values(by='checkin_week', ascending=True).head(20)

,ave_price
checkin_week,
2026-06-08,431.290555
2026-06-15,347.953787
2026-06-22,337.569286
2026-06-29,214.355442
2026-07-06,287.136142
2026-07-13,215.344712
2026-07-20,255.236954
2026-07-27,188.108878
2026-08-03,195.741665
